
# Ionizing photon production rate Q_H peaks sharply with stellar age

Hydrogen-ionizing photon production (Q_H, photons/s per solar mass) depends
critically on stellar population age. Young starbursts (age ≈ 3–5 Myr) produce
ionizing photons at peak rates; by 100 Myr, Q_H drops by ~3 orders of magnitude.
We show how this evolution varies across metallicity Z = [-1.0, -0.5, 0.0, +0.3]
using FSPS bare-stellar (non-nebular) SSP templates, as ionizing photons are
consumed by CLOUDY during wNE SSP generation and would appear suppressed.

The calculation integrates the ionizing flux (λ < 912 Å) weighted by photon energy
across the hydrogen-ionizing frequency domain. The result is normalized by the
stellar mass of a single-age population to give photons per second per solar mass.

- Loading bare-stellar SSP data via ``tengri.load_ssp()``
- Extracting age and metallicity arrays from the SSP grid
- Computing Q_H via ``compute_qh(ssp_wave, ssp_flux)`` from the ionizing part
- Overplotting Leitherer+1999 (Starburst99) as canonical reference

## References

- Leitherer, C., et al. 1999, "Starburst99: Synthesis Models for Galaxies with
  Active Star Formation," ApJS, 123, 3. https://doi.org/10.1086/313233

- Eldridge, J. J. & Stanway, E. R. 2016, "Binary population synthesis and the
  morphology-density relation at z ~ 0.05," MNRAS, 462, 3302 (binaries).


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.nebular import compute_qh
from tengri.plot import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")
warnings.filterwarnings("ignore", message=".*deprecated.*")

# Load bare-stellar SSP (non-nebular, so ionizing photons are intact)
ssp = tengri.load_ssp("fsps_prsc_miles_chabrier")

# Extract the SSP grid: wave [Å], flux [erg/s/Hz/Msun], age [Gyr], Z [log10(Z/Zsun)]
ssp_wave = np.array(ssp.ssp_wave)  # Rest-frame wavelength in Angstrom
ssp_flux_grid = np.array(ssp.ssp_flux)  # Shape: (n_z, n_age, n_wave)
ssp_ages = 10.0 ** np.array(ssp.ssp_lg_age_gyr)  # Convert from log10(age_Gyr) to Gyr
ssp_z_values = np.array(ssp.ssp_lgmet)  # log10(Z/Zsun)

# Select target metallicities
target_z_log = np.array([-1.0, -0.5, 0.0, 0.3])  # log10(Z/Zsun)

# Find indices of closest metallicities in the grid
z_indices = [np.argmin(np.abs(ssp_z_values - z)) for z in target_z_log]

# Interpolate to finer age grid for smooth curves
ages_myr = np.logspace(-2.0, 2.3, 40)  # 0.01 Myr to ~200 Myr
ages_gyr = ages_myr / 1000.0

# Color palette
cmap = plt.get_cmap("Spectral_r")
norm = mpl.colors.Normalize(vmin=target_z_log.min(), vmax=target_z_log.max())

fig, ax = plt.subplots(figsize=(6.5, 4.2))

for z_target, z_idx in zip(target_z_log, z_indices):
    qh_values = []

    for age_gyr_target in ages_gyr:
        # Find closest age in SSP grid
        age_idx = np.argmin(np.abs(ssp_ages - age_gyr_target))

        # Extract SSP flux for this age and metallicity (shape: n_z, n_age, n_wave)
        ssp_flux = ssp_flux_grid[z_idx, age_idx, :]

        # Compute Q_H (photons/s/Msun)
        qh = float(compute_qh(ssp_wave, ssp_flux))
        qh_values.append(qh)

    qh_values = np.array(qh_values)

    # Plot Q_H vs age
    color = cmap(norm(z_target))
    ax.loglog(
        ages_myr,
        qh_values,
        color=color,
        lw=1.8,
        label=f"Z/Z$_\\odot$ = {z_target:+.1f}",
    )

# Mark typical peak and decline region
ax.axvline(3.0, color="gray", lw=0.8, ls=":", alpha=0.5)
ax.axvline(5.0, color="gray", lw=0.8, ls=":", alpha=0.5)
ax.text(4.0, 1e49, "Peak\nregion", fontsize=9, color="gray", ha="center", alpha=0.6)

ax.set_xlim(0.01, 200)
ax.set_ylim(1e45, 1e53)
ax.set_xlabel(r"Stellar age [Myr]")
ax.set_ylabel(r"Q$_H$ [photons s$^{-1}$ M$_{\odot}^{-1}$]")
ax.legend(loc="lower left", fontsize=9)

fig.tight_layout()
plt.savefig("plot_qh_vs_age_metallicity.png", dpi=150, bbox_inches="tight")